# LexisNexis Fraud Detection - Task 5: Hyperparameter Tuning

Tunes the key settings of each model (logistic regression, random forest, gradient boosting) to get the best PR-AUC. Setup (data, features, preprocessing) is copied from Task 1, including `NewPolicy`, so every model sees the same 129 features.

In [1]:
!rm -rf LexisNexis-1A-detecting-fraudulent-insurance-claims
!git clone -q https://github.com/Break-Through-Tech/LexisNexis-1A-detecting-fraudulent-insurance-claims.git
%cd /content/LexisNexis-1A-detecting-fraudulent-insurance-claims

/content/LexisNexis-1A-detecting-fraudulent-insurance-claims


In [2]:
import json

import numpy as np
import pandas as pd
from scipy.stats import loguniform

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score

pd.set_option("display.max_columns", None)

In [3]:
RANDOM_STATE = 42
N_ITER = 30      # settings combinations tried per model
CV_FOLDS = 5     # cross-validation folds inside 1994
N_BOOT = 1000    # bootstrap resamples for the noise check at the end

In [4]:
# Task 1 saves both of these so every M2 notebook uses identical data and
# identical features. Copying the feature code between notebooks caused real
# drift earlier: Task 2's copy was missing NewPolicy and Task 3's was not.
df = pd.read_csv("data/fraud_oracle_engineered.csv")

with open("data/model_features.json") as f:
    MODEL_FEATURES = json.load(f)

TARGET = "FraudFound_P"

train = df[df["Year"] == 1994].copy()
val   = df[df["Year"] == 1995].copy()
# 1996 is deliberately not loaded. It gets opened once, in Task 6.

X_train, y_train = train[MODEL_FEATURES], train[TARGET]
X_val,   y_val   = val[MODEL_FEATURES],   val[TARGET]

print("Features:", len(MODEL_FEATURES))
print("Train (1994):", X_train.shape, "fraud cases:", int(y_train.sum()))
print("Validation (1995):", X_val.shape, "fraud cases:", int(y_val.sum()))

Features: 24
Train (1994): (6141, 24) fraud cases: 409
Validation (1995): (5195, 24) fraud cases: 301


In [5]:
numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

# only group columns that are in the feature list. PolicyType was dropped at
# the Challenge Advisor's request, so this guard stops the ColumnTransformer
# looking for a column that is not there.
grouped_categories = [c for c in ["Make", "PolicyType"]
                      if c in categorical_features]
regular_categories = [c for c in categorical_features
                      if c not in grouped_categories]


def make_preprocessor(dense=False):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    grouped_category_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                 min_frequency=0.01)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, regular_categories),
        ("grouped_categorical", grouped_category_pipeline, grouped_categories),
    ], sparse_threshold=0 if dense else 0.3)


def build_pipeline(model, dense=False):
    """Preprocessing + model in one object, so .fit() always fits the
    preprocessing on the training data only, including inside each CV fold."""
    return Pipeline([("prep", make_preprocessor(dense)), ("clf", model)])


print("Grouped:", grouped_categories)
print("Processed features:",
      make_preprocessor().fit(X_train).transform(X_train).shape[1])  # expect 88

Grouped: ['Make']
Processed features: 88


## Evaluation function (same as Tasks 2 and 3)

Kept identical so the rows here line up with the earlier notebooks: precision, recall and F1 at a 0.5 threshold, PR-AUC, and the share of fraud captured in the top 10% / 20% highest-scored claims.

In [6]:
CAPACITY = 0.20   # how much of the claim book a review team can get through


def evaluate(name, model, X_tr, y_tr, X_ev, y_ev):
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_ev)[:, 1]

    pred_half = (proba >= 0.5).astype(int)
    thresh = np.quantile(proba, 1 - CAPACITY)
    pred_cap = (proba >= thresh).astype(int)

    y_sorted = y_ev.to_numpy()[proba.argsort()[::-1]]
    cap10 = y_sorted[:int(len(y_sorted) * 0.10)].sum() / y_sorted.sum()
    cap20 = y_sorted[:int(len(y_sorted) * 0.20)].sum() / y_sorted.sum()

    return {
        "Approach": name,
        "PR-AUC": average_precision_score(y_ev, proba),
        "Caught top 10%": cap10,
        "Caught top 20%": cap20,
        "Precision @cap": precision_score(y_ev, pred_cap, zero_division=0),
        "Recall @cap": recall_score(y_ev, pred_cap),
        "F1 @cap": f1_score(y_ev, pred_cap),
        "Precision @0.5": precision_score(y_ev, pred_half, zero_division=0),
        "Recall @0.5": recall_score(y_ev, pred_half),
        "F1 @0.5": f1_score(y_ev, pred_half),
    }

## How the tuning works

For each model, RandomizedSearchCV picks N_ITER random combinations of settings from the ranges defined below. Each combination is scored with 5-fold cross-validation on 1994: the 1994 claims are split into 5 chunks (each keeping the 6.7% fraud rate), the model trains on 4 chunks and gets scored on the 5th, and this rotates until every chunk has been scored once. The 5 scores are averaged, and the combination with the best average PR-AUC wins. Top 10% and top 20% fraud capture are tracked too.

In [7]:
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def capture_scorer(frac):
    """Scorer for the business metric: share of all fraud found in the top `frac` highest-scored claims."""
    def score(estimator, X, y):
        proba = estimator.predict_proba(X)[:, 1]
        y_arr = np.asarray(y)
        k = int(len(y_arr) * frac)
        top = np.argsort(proba)[::-1][:k]
        return y_arr[top].sum() / y_arr.sum()
    return score

SCORING = {
    "pr_auc": "average_precision",   # scikit-learn's name for PR-AUC
    "top10": capture_scorer(0.10),
    "top20": capture_scorer(0.20),
}

def clean_params(params):
    """Strip the 'clf__' prefix and turn numpy numbers into plain Python numbers."""
    return {k.replace("clf__", ""): (v.item() if hasattr(v, "item") else v) for k, v in params.items()}

def run_search(name, model, param_space, dense=False):
    search = RandomizedSearchCV(
        build_pipeline(model, dense),
        param_distributions=param_space,
        n_iter=N_ITER,
        scoring=SCORING,
        refit="pr_auc",              # pick the winner by PR-AUC
        cv=cv,
        return_train_score=True,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    search.fit(X_train, y_train)     # 1994 only
    spread = search.cv_results_["std_test_pr_auc"][search.best_index_]
    print(f"{name}: best CV PR-AUC {search.best_score_:.4f} (+/- {spread:.4f} across folds)")
    print("Best settings:", clean_params(search.best_params_))
    return search

def show_top(search, k=5):
    """The k best settings combinations, ranked by cross-validated PR-AUC."""
    res = pd.DataFrame(search.cv_results_)
    params = [c for c in res.columns if c.startswith("param_")]
    cols = ["mean_test_pr_auc", "std_test_pr_auc", "mean_train_pr_auc",
            "mean_test_top10", "mean_test_top20"] + params
    out = res.sort_values("mean_test_pr_auc", ascending=False)[cols].head(k)
    out.columns = [c.replace("param_clf__", "") for c in out.columns]
    return out.round(4).reset_index(drop=True)

## 1. Logistic regression

The main changer is C, which controls regularization. A small C pulls the weights toward zero which means we'd have a simpler model and less overfitting, and a large C lets the weights grow to fit 1994 closely. It's searched on a log scale from 0.001 to 100, since its effect multiplies.

In [8]:
lr_space = {"clf__C": loguniform(1e-3, 1e2)}

lr_search = run_search(
    "Logistic regression",
    LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    lr_space,
)
show_top(lr_search)

Logistic regression: best CV PR-AUC 0.2033 (+/- 0.0223 across folds)
Best settings: {'C': 0.008260808399079604}


,mean_test_pr_auc,std_test_pr_auc,mean_train_pr_auc,mean_test_top10,mean_test_top20,C
0,0.2033,0.0223,0.2216,0.3055,0.5549,0.0083
1,0.2032,0.0199,0.2194,0.3080,0.5524,0.0050
2,0.2031,0.0222,0.2216,0.3080,0.5524,0.0081
3,0.2026,0.0222,0.2222,0.3080,0.5549,0.0100
4,0.2025,0.0203,0.2200,0.3080,0.5500,0.0060


## 2. Random Forest


We will use `GridSearchCV` to tune key hyperparameters of the Random Forest. We'll optimize for `average_precision` (PR-AUC) since it is a robust metric for highly imbalanced fraud detection tasks.

In [9]:
from sklearn.model_selection import GridSearchCV

# The grid now goes down to 1, the fully grown forest. The previous grid
# started at 10 and picked 10, the smallest value it was offered, which is a
# sign the optimum sits outside the grid. On the 24 column feature set, Task 3
# found leaf size 1 to be the best setting.
rf_grid = {
    "clf__n_estimators": [300, 500],
    "clf__max_depth": [None, 20],
    "clf__min_samples_leaf": [1, 2, 5, 10, 30],
    "clf__max_features": ["sqrt", "log2"],
}

rf_search = GridSearchCV(
    build_pipeline(RandomForestClassifier(class_weight="balanced",
                                          random_state=RANDOM_STATE, n_jobs=1)),
    param_grid=rf_grid,
    scoring=SCORING,
    refit="pr_auc",
    cv=cv,
    return_train_score=True,
    n_jobs=-1,
)
rf_search.fit(X_train, y_train)

print(f"Random forest: best CV PR-AUC {rf_search.best_score_:.4f}")
print("Best settings:", clean_params(rf_search.best_params_))
show_top(rf_search)

Random forest: best CV PR-AUC 0.2414
Best settings: {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 10, 'n_estimators': 500}


,mean_test_pr_auc,std_test_pr_auc,mean_train_pr_auc,mean_test_top10,mean_test_top20,max_depth,max_features,min_samples_leaf,n_estimators
0,0.2414,0.0335,0.6049,0.3617,0.5573,20,sqrt,10,500
1,0.2404,0.0328,0.6057,0.3592,0.5622,None,sqrt,10,500
2,0.2404,0.0326,0.6010,0.3617,0.5646,20,sqrt,10,300
3,0.2396,0.0332,0.3708,0.3714,0.5915,None,log2,30,300
4,0.2396,0.0331,0.3708,0.3714,0.5891,20,log2,30,300


## 3. Gradient boosting

The Challenge Advisor suggested gradient boosting might not be necessary, on
the basis that a tuned random forest is a solid final model. We ran it anyway,
and on this data it came out ahead of the forest by a clear margin, so it stays
in the comparison. Settings tuned:

In [10]:
xgb_search = None
gb_space = {
    "clf__learning_rate": loguniform(0.01, 0.2),
    "clf__max_iter": [100, 200, 400, 600],
    "clf__max_leaf_nodes": [7, 15, 31],
    "clf__max_depth": [None, 3, 5],
    "clf__min_samples_leaf": [20, 50, 100, 200],
    "clf__l2_regularization": [0.0, 0.1, 1.0, 10.0],
    "clf__max_features": [0.3, 0.5, 1.0],
}

gb_search = run_search(
    "Gradient boosting",
    HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False,
                                   random_state=RANDOM_STATE),
    gb_space,
    dense=True,
)
show_top(gb_search)

Gradient boosting: best CV PR-AUC 0.2638 (+/- 0.0322 across folds)
Best settings: {'l2_regularization': 0.1, 'learning_rate': 0.029266761285490727, 'max_depth': None, 'max_features': 1.0, 'max_iter': 100, 'max_leaf_nodes': 15, 'min_samples_leaf': 20}


,mean_test_pr_auc,std_test_pr_auc,mean_train_pr_auc,mean_test_top10,mean_test_top20,l2_regularization,learning_rate,max_depth,max_features,max_iter,max_leaf_nodes,min_samples_leaf
0,0.2638,0.0322,0.3958,0.3642,0.6062,0.1,0.0293,None,1.0,100,15,20
1,0.2444,0.0148,0.8985,0.3448,0.5769,1.0,0.0411,None,1.0,200,31,20
2,0.2437,0.0263,0.5742,0.3519,0.5939,1.0,0.0831,5,1.0,100,31,20
3,0.2406,0.0363,0.4160,0.3641,0.5841,10.0,0.0115,None,0.3,200,31,20
4,0.2380,0.0309,0.4550,0.3616,0.5915,10.0,0.0291,None,0.3,100,31,20


## Results on the 1995 validation set

Each model is retrained on all of 1994 and scored once on 1995 with the team's `evaluate()`. The untuned reference rows should match Task 3's table (LR + class_weight PR-AUC 0.0948, RF with Task 3 settings 0.1098, RF from Task 3's grid search 0.1100). If they don't, something in your setup differs from the team's.

In [11]:
candidates = {
    # untuned references from earlier tasks, on the current feature list
    "LR, no handling (Task 2 best)": build_pipeline(
        LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "LR, class weighted": build_pipeline(
        LogisticRegression(class_weight="balanced", max_iter=1000,
                           random_state=RANDOM_STATE)),
    "RF, fully grown (Task 3 winner)": build_pipeline(
        RandomForestClassifier(n_estimators=500, class_weight="balanced",
                               random_state=RANDOM_STATE, n_jobs=-1)),
    "GB, default settings": build_pipeline(
        HistGradientBoostingClassifier(class_weight="balanced",
                                       random_state=RANDOM_STATE),
        dense=True),
    # tuned in this notebook
    "LR tuned": lr_search.best_estimator_,
    "RF tuned": rf_search.best_estimator_,
    "GB tuned": gb_search.best_estimator_,
}

results, val_probs = [], {}
for name, model in candidates.items():
    results.append(evaluate(name, model, X_train, y_train, X_val, y_val))
    val_probs[name] = model.predict_proba(X_val)[:, 1]

results_df = pd.DataFrame(results).set_index("Approach").round(4)

print("RANKING QUALITY.\n")
display(results_df[["PR-AUC", "Caught top 10%", "Caught top 20%"]]
        .sort_values("PR-AUC", ascending=False))

print(f"\n\nAT A CUTOFF THAT FLAGS THE TOP {int(CAPACITY*100)}% OF CLAIMS.\n")
display(results_df[["Precision @cap", "Recall @cap", "F1 @cap"]]
        .sort_values("F1 @cap", ascending=False))

RANKING QUALITY.



,PR-AUC,Caught top 10%,Caught top 20%
Approach,,,
GB tuned,0.1901,0.2857,0.5116
"GB, default settings",0.1770,0.2791,0.4751
"RF, fully grown (Task 3 winner)",0.1556,0.3223,0.5415
"LR, no handling (Task 2 best)",0.1488,0.2824,0.5216
RF tuned,0.1463,0.2791,0.5150
"LR, class weighted",0.1393,0.2724,0.5083
LR tuned,0.1317,0.2791,0.4784




AT A CUTOFF THAT FLAGS THE TOP 20% OF CLAIMS.



,Precision @cap,Recall @cap,F1 @cap
Approach,,,
"RF, fully grown (Task 3 winner)",0.1536,0.5415,0.2394
"LR, no handling (Task 2 best)",0.1511,0.5216,0.2343
RF tuned,0.1492,0.5150,0.2313
GB tuned,0.1482,0.5116,0.2299
"LR, class weighted",0.1473,0.5083,0.2284
LR tuned,0.1386,0.4784,0.2149
"GB, default settings",0.1376,0.4751,0.2134


## Noise check: are the differences real?

1995 has only about 300 fraud cases, so PR-AUC moves around by chance. This cell resamples the 1995 claims 1,000 times (a bootstrap) and recomputes PR-AUC on each resample.

- 95% low / 95% high: the range PR-AUC falls in for 95% of resamples.
- Diff vs LR: PR-AUC minus the untuned logistic regression's PR-AUC, with its own 95% range.
- Beats LR: share of resamples where the model beat the untuned logistic regression.

If the 95% range of the difference includes 0, we can't confidently call that model better.

In [12]:
BASELINE = "LR, no handling (Task 2 best)"

rng = np.random.default_rng(RANDOM_STATE)
y_val_arr = y_val.to_numpy()
boot_idx = rng.integers(0, len(y_val_arr), size=(N_BOOT, len(y_val_arr)))


def boot_pr_auc(proba):
    return np.array([average_precision_score(y_val_arr[i], proba[i])
                     for i in boot_idx])


base_point = average_precision_score(y_val_arr, val_probs[BASELINE])
base_boot = boot_pr_auc(val_probs[BASELINE])

rows, boot_store = [], {}
for name, proba in val_probs.items():
    point = average_precision_score(y_val_arr, proba)
    s = boot_pr_auc(proba)
    boot_store[name] = s
    d = s - base_boot
    is_base = name == BASELINE
    rows.append({
        "Approach": name,
        "PR-AUC": point,
        "95% low": np.percentile(s, 2.5),
        "95% high": np.percentile(s, 97.5),
        "Diff vs baseline": np.nan if is_base else point - base_point,
        "Diff 95% low": np.nan if is_base else np.percentile(d, 2.5),
        "Diff 95% high": np.nan if is_base else np.percentile(d, 97.5),
        "Beats baseline": np.nan if is_base else (d > 0).mean(),
    })

noise_df = pd.DataFrame(rows).set_index("Approach").round(4)
display(noise_df)

# Head to head: the comparison above only tests each model against the
# baseline. For Task 6 the question is whether the best model is really
# better than the runner up, which needs its own test.
ranked = noise_df["PR-AUC"].sort_values(ascending=False)
best_name, second_name = ranked.index[0], ranked.index[1]
d2 = boot_store[best_name] - boot_store[second_name]

print(f"\nHead to head: {best_name} vs {second_name}")
print(f"  difference in PR-AUC: {ranked.iloc[0] - ranked.iloc[1]:+.4f}")
print(f"  95% range of the difference: "
      f"{np.percentile(d2, 2.5):+.4f} to {np.percentile(d2, 97.5):+.4f}")
print(f"  ahead in {(d2 > 0).mean():.1%} of the {N_BOOT} resamples")
if np.percentile(d2, 2.5) <= 0:
    print("  The range includes 0, so this is not a clear win. "
          "Report them as close rather than picking a winner on this alone.")
else:
    print("  The range stays above 0, so this holds up as a real difference.")

,PR-AUC,95% low,95% high,Diff vs baseline,Diff 95% low,Diff 95% high,Beats baseline
Approach,,,,,,,
"LR, no handling (Task 2 best)",0.1488,0.1273,0.1758,NaN,NaN,NaN,NaN
"LR, class weighted",0.1393,0.1194,0.1653,-0.0095,-0.0203,0.0004,0.031
"RF, fully grown (Task 3 winner)",0.1556,0.1328,0.1846,0.0068,-0.0150,0.0285,0.740
"GB, default settings",0.1770,0.1470,0.2177,0.0283,-0.0028,0.0615,0.959
LR tuned,0.1317,0.1142,0.1549,-0.0171,-0.0320,-0.0051,0.004
RF tuned,0.1463,0.1263,0.1709,-0.0025,-0.0200,0.0138,0.390
GB tuned,0.1901,0.1576,0.2284,0.0413,0.0080,0.0772,0.993



Head to head: GB tuned vs GB, default settings
  difference in PR-AUC: +0.0131
  95% range of the difference: -0.0093 to +0.0320
  ahead in 87.5% of the 1000 resamples
  The range includes 0, so this is not a clear win. Report them as close rather than picking a winner on this alone.


In [14]:
# PR-AUC and the capture rates disagree about which model is best, so the
# capture rate needs its own bootstrap. PR-AUC averages over every possible
# cutoff; we only ever operate in the top 10 to 20%.
def boot_capture(proba, frac=0.20):
    k = int(len(y_val_arr) * frac)
    out = []
    for i in boot_idx:
        yb, pb = y_val_arr[i], proba[i]
        top = np.argsort(pb)[::-1][:k]
        out.append(yb[top].sum() / yb.sum())
    return np.array(out)


cap_store = {name: boot_capture(p) for name, p in val_probs.items()}

cap_rows = []
for name, s in cap_store.items():
    cap_rows.append({
        "Approach": name,
        "Caught top 20%": results_df.loc[name, "Caught top 20%"],
        "95% low": np.percentile(s, 2.5),
        "95% high": np.percentile(s, 97.5),
    })
print("Top 20% capture with 95% ranges:\n")
display(pd.DataFrame(cap_rows).set_index("Approach").round(4)
        .sort_values("Caught top 20%", ascending=False))

# The call Task 6 has to make
A = "GB tuned"
B = "RF, fully grown (Task 3 winner)"

for label, store in [("PR-AUC", boot_store),
                     ("top 20% capture", cap_store)]:
    d = store[A] - store[B]
    lo, hi = np.percentile(d, 2.5), np.percentile(d, 97.5)
    if label == "PR-AUC":
        point = results_df.loc[A, "PR-AUC"] - results_df.loc[B, "PR-AUC"]
    else:
        point = (results_df.loc[A, "Caught top 20%"]
                 - results_df.loc[B, "Caught top 20%"])
    print(f"\n{A} minus {B}, on {label}")
    print(f"  difference: {point:+.4f}")
    print(f"  95% range:  {lo:+.4f} to {hi:+.4f}")
    print(f"  {A} ahead in {(d > 0).mean():.1%} of resamples")
    print("  -> clear difference" if (lo > 0 or hi < 0)
          else "  -> NOT clear, the range includes 0")

Top 20% capture with 95% ranges:



,Caught top 20%,95% low,95% high
Approach,,,
"RF, fully grown (Task 3 winner)",0.5415,0.4813,0.5884
"LR, no handling (Task 2 best)",0.5216,0.4674,0.5714
RF tuned,0.5150,0.4625,0.5683
GB tuned,0.5116,0.4539,0.5648
"LR, class weighted",0.5083,0.4549,0.5611
LR tuned,0.4784,0.4334,0.5352
"GB, default settings",0.4751,0.4192,0.5326



GB tuned minus RF, fully grown (Task 3 winner), on PR-AUC
  difference: +0.0345
  95% range:  +0.0057 to +0.0633
  GB tuned ahead in 98.7% of resamples
  -> clear difference

GB tuned minus RF, fully grown (Task 3 winner), on top 20% capture
  difference: -0.0299
  95% range:  -0.0724 to +0.0250
  GB tuned ahead in 17.0% of resamples
  -> NOT clear, the range includes 0


## Save results for Task 6


In [13]:
searches = {
    "logistic_regression": lr_search,
    "random_forest": rf_search,
    "gradient_boosting": gb_search,
}

tuned = {
    name: {"best_params": clean_params(s.best_params_),
           "cv_pr_auc_1994": round(float(s.best_score_), 4)}
    for name, s in searches.items()
}

with open("data/m2_task4_5_tuned_params.json", "w") as f:
    json.dump(tuned, f, indent=2)

results_df.join(noise_df.drop(columns="PR-AUC")).to_csv(
    "data/m2_task4_5_validation_results.csv")

print(json.dumps(tuned, indent=2))
print("\nSaved to data/m2_task4_5_tuned_params.json")
print("Saved to data/m2_task4_5_validation_results.csv")
print("\nRanked by validation PR-AUC:")
print(results_df.sort_values("PR-AUC", ascending=False)
      [["PR-AUC", "Caught top 10%", "Caught top 20%"]])

{
  "logistic_regression": {
    "best_params": {
      "C": 0.008260808399079604
    },
    "cv_pr_auc_1994": 0.2033
  },
  "random_forest": {
    "best_params": {
      "max_depth": 20,
      "max_features": "sqrt",
      "min_samples_leaf": 10,
      "n_estimators": 500
    },
    "cv_pr_auc_1994": 0.2414
  },
  "gradient_boosting": {
    "best_params": {
      "l2_regularization": 0.1,
      "learning_rate": 0.029266761285490727,
      "max_depth": null,
      "max_features": 1.0,
      "max_iter": 100,
      "max_leaf_nodes": 15,
      "min_samples_leaf": 20
    },
    "cv_pr_auc_1994": 0.2638
  }
}

Saved to data/m2_task4_5_tuned_params.json
Saved to data/m2_task4_5_validation_results.csv

Ranked by validation PR-AUC:
                                 PR-AUC  Caught top 10%  Caught top 20%
Approach                                                               
GB tuned                         0.1901          0.2857          0.5116
GB, default settings             0.1770          0

## Summary

Tuned logistic regression, random forest and gradient boosting, each scored by
5-fold cross-validated PR-AUC inside the 1994 training set, then evaluated once
on 1995. Preprocessing ran inside every pipeline so it refits on each fold, and
the 1996 test year was not used. All models ran on the 24 column feature list
from Task 1.

This notebook also covers Task 4, since gradient boosting is built and tuned here.

### Results on 1995

| Model | PR-AUC | Caught top 10% | Caught top 20% |
|---|---|---|---|
| **GB tuned** | **0.1901** | 0.2857 | 0.5116 |
| GB default | 0.1770 | 0.2791 | 0.4751 |
| RF fully grown (Task 3 winner) | 0.1556 | **0.3223** | **0.5415** |
| LR no handling (Task 2 best) | 0.1488 | 0.2824 | 0.5216 |
| RF tuned | 0.1463 | 0.2791 | 0.5150 |
| LR class weighted | 0.1393 | 0.2724 | 0.5083 |
| LR tuned | 0.1317 | 0.2791 | 0.4784 |

The two metrics disagree. Gradient boosting leads on PR-AUC by a wide margin,
and the random forest leads on both capture rates by about three points. That
is not a contradiction: PR-AUC averages performance across every possible
cutoff, while the capture rates only look at the top slice we would actually
review. A model can win the average and lose the region that matters.

### The bootstrap settles part of it

1,000 resamples of the 1995 claims, measured against the untuned logistic
regression.

| Model | Diff vs baseline | 95% range | Beats baseline |
|---|---|---|---|
| **GB tuned** | **+0.0413** | **+0.0080 to +0.0772** | **99.3%** |
| GB default | +0.0283 | -0.0028 to +0.0615 | 95.9% |
| RF fully grown | +0.0068 | -0.0150 to +0.0285 | 74.0% |
| RF tuned | -0.0025 | -0.0200 to +0.0138 | 39.0% |
| LR class weighted | -0.0095 | -0.0203 to +0.0004 | 3.1% |
| LR tuned | -0.0171 | -0.0320 to -0.0051 | 0.4% |

Tuned gradient boosting is the only model whose range stays entirely above
zero. Every other model, the random forest included, has a range that crosses
zero, which means we cannot claim it beats plain logistic regression.

Head to head, tuned gradient boosting against the fully grown forest:

- **On PR-AUC:** +0.0345, 95% range +0.0057 to +0.0633, ahead in 98.7% of
  resamples. A clear difference.
- **On top 20% capture:** -0.0299, 95% range -0.0724 to +0.0250, ahead in only
  17% of resamples. Not clear, the range includes zero.

So gradient boosting's advantage is real but happens largely outside the slice
we operate in. It orders the whole book better. It does not demonstrably put
more fraud in the top 20%.

There is a statistical reason the two tests differ in confidence. PR-AUC uses
all 5,195 validation claims. Top 20% capture only looks at about 1,039 claims
holding roughly 60 fraud cases, so it is a far noisier measurement. Every
model's capture interval in this notebook sits somewhere between 0.42 and
0.59, and they all overlap. On capture alone, no model here can be separated
from any other.
